# Biosignatures in Exoplanet Atmospheres — Demo Pipeline
**Week 2 Deliverable** | End-to-end transit spectroscopy simulation

This notebook demonstrates the full pipeline:
1. Load the three toy atmosphere templates
2. Visualize template spectra and biosignature features
3. Model JWST NIRSpec instrument response
4. Simulate TRAPPIST-1e observations (10 transits)
5. Detection horizon: SNR vs. distance
6. Cloud fraction sensitivity analysis

> **To run:** `jupyter notebook notebooks/demo_pipeline.ipynb` from project root.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))
os.chdir('..')

import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from atmosphere_templates import (
    get_default_templates, TemplateGrid, default_wavelength_grid,
    build_earth_like_template, build_high_co2_template, build_reduced_o2_high_ch4_template
)
from instrument_model import load_jwst_nirspec
from observation_sim import ObservationSimulator, PlanetSystem

plt.rcParams.update({'figure.dpi': 130, 'axes.spines.top': False, 'axes.spines.right': False})
COLORS = {'earth_like': '#1565C0', 'high_co2': '#BF360C', 'reduced_o2_high_ch4': '#2E7D32'}
LABELS = {'earth_like': 'Earth-like (O2+H2O+CO2)', 'high_co2': 'High CO2 (Venus-analog)', 'reduced_o2_high_ch4': 'Reduced O2/High CH4 (Archean)'}
print('Setup complete.')

## 1. Load Atmosphere Templates

In [ ]:
STAR_RS = 0.1192
PLANET_RE = 0.92
templates = get_default_templates(star_radius_rs=STAR_RS, planet_radius_re=PLANET_RE)
for name, t in templates.items():
    base = t.parameters['base_depth_ppm']
    peak = t.transit_depth_ppm.max()
    print(f'{name:30s}  base={base:.0f} ppm  peak={peak:.0f} ppm  delta={peak-base:.0f} ppm')

## 2. Template Spectra + Biosignature Feature Map

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True)
fig.suptitle('Toy Atmosphere Templates — Transit Depth Spectra', fontsize=14, fontweight='bold')
features = [(0.762,'O2 A','top'),(0.940,'H2O','bot'),(1.140,'H2O','top'),(1.380,'H2O','bot'),
            (1.600,'CO2','top'),(1.670,'CH4','bot'),(1.870,'H2O','top'),(2.300,'CH4','bot'),(4.300,'CO2','top')]
for ax, (name, tmpl) in zip(axes, templates.items()):
    wl = tmpl.wavelengths_um
    depth = tmpl.transit_depth_ppm
    base = tmpl.parameters['base_depth_ppm']
    ax.fill_between(wl, base, depth, alpha=0.15, color=COLORS[name])
    ax.plot(wl, depth, color=COLORS[name], lw=1.8, label=LABELS[name])
    ax.axhline(base, color='gray', lw=0.8, ls='--', alpha=0.6)
    y_range = depth.max() - depth.min()
    for feat_wl, feat_label, side in features:
        ax.axvline(feat_wl, color='gray', lw=0.5, ls=':', alpha=0.4)
        y_pos = depth.max() + y_range*0.08 if side=='top' else depth.min() - y_range*0.14
        ax.text(feat_wl, y_pos, feat_label, ha='center', fontsize=7, color='#666', rotation=90)
    ax.set_ylabel('Transit Depth (ppm)')
    ax.legend(loc='upper right', fontsize=9)
    ax.set_xlim(0.6, 5.3)
axes[-1].set_xlabel('Wavelength (um)')
os.makedirs('results', exist_ok=True)
plt.tight_layout()
plt.savefig('results/fig01_template_spectra.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved results/fig01_template_spectra.png')

## 3. JWST NIRSpec Instrument Response

In [ ]:
jwst = load_jwst_nirspec()
print(jwst.summary())
wl = default_wavelength_grid()
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('JWST NIRSpec Prism — Instrument Model', fontsize=13, fontweight='bold')
tp = jwst.throughput(wl)
axes[0].plot(wl, tp*100, color='#1A237E', lw=2)
axes[0].fill_between(wl, 0, tp*100, alpha=0.15, color='#1A237E')
axes[0].set_xlabel('Wavelength (um)'); axes[0].set_ylabel('Throughput (%)')
axes[0].set_title('End-to-end throughput'); axes[0].set_xlim(0.6, 5.3)
rate_t1  = jwst.stellar_photon_rate(wl, 11.35, 2566)
rate_lhs = jwst.stellar_photon_rate(wl, 9.61,  3216)
axes[1].semilogy(wl, rate_t1, color='#880E4F', lw=1.8, label='TRAPPIST-1 (J=11.4)')
axes[1].semilogy(wl, rate_lhs, color='#E65100', lw=1.8, ls='--', label='LHS 1140 (J=9.6)')
axes[1].set_xlabel('Wavelength (um)'); axes[1].set_ylabel('Photon rate (ph/s/bin)')
axes[1].set_title('Stellar photon rate at detector')
axes[1].legend(fontsize=9); axes[1].set_xlim(0.6, 5.3)
plt.tight_layout()
plt.savefig('results/fig02_instrument_model.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved results/fig02_instrument_model.png')

## 4. Simulated TRAPPIST-1e Observations (10 Transits)

In [ ]:
planet = PlanetSystem.trappist1e()
sim = ObservationSimulator(instrument=jwst, rng=np.random.default_rng(42))
templates_t1 = get_default_templates(star_radius_rs=planet.star_radius_rs, planet_radius_re=planet.planet_radius_re)
results_10t = {}
for name, tmpl in templates_t1.items():
    r = sim.simulate(planet, tmpl, n_transits=10, verbose=True)
    results_10t[name] = r
    print(r.summary())

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
fig.suptitle('Simulated JWST NIRSpec Observations — TRAPPIST-1e (10 transits)', fontsize=14, fontweight='bold')
for ax, (name, result) in zip(axes, results_10t.items()):
    wl = result.wavelengths_um
    mask = (wl >= 0.6) & (wl <= 5.3) & (result.noise_ppm < 5000)
    ax.plot(wl[mask], result.true_depth_ppm[mask], color=COLORS[name], lw=2.0, label='True spectrum', zorder=3)
    ax.errorbar(wl[mask][::3], result.observed_depth_ppm[mask][::3], yerr=result.noise_ppm[mask][::3],
                fmt='o', color=COLORS[name], alpha=0.45, ms=2.5, lw=0.8, capsize=1.5, label='Simulated obs.', zorder=2)
    ax.fill_between(wl[mask], result.true_depth_ppm[mask]-result.noise_ppm[mask],
                    result.true_depth_ppm[mask]+result.noise_ppm[mask], alpha=0.10, color=COLORS[name])
    det = 'DETECTED' if result.is_detected else 'not detected'
    ax.set_title(f'{LABELS[name]}   |   Broadband SNR = {result.detection_snr:.1f}s  [{det}]',
                 fontsize=10, color=COLORS[name])
    ax.set_ylabel('Transit Depth (ppm)'); ax.legend(loc='upper right', fontsize=8); ax.set_xlim(0.6, 5.3)
axes[-1].set_xlabel('Wavelength (um)')
plt.tight_layout()
plt.savefig('results/fig03_simulated_observations.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved results/fig03_simulated_observations.png')

## 5. Detection Horizon: SNR vs. Distance

In [ ]:
distances_pc = np.array([3, 5, 8, 10, 12.4, 15, 20, 25, 30, 40, 50])
wl = default_wavelength_grid()
horizon_snrs = {}
for name, builder in [('earth_like', build_earth_like_template), ('high_co2', build_high_co2_template), ('reduced_o2_high_ch4', build_reduced_o2_high_ch4_template)]:
    tmpl = builder(wl, star_radius_rs=0.2, planet_radius_re=1.0)
    _, snrs = sim.detection_horizon(tmpl, distances_pc, n_transits=10)
    horizon_snrs[name] = snrs
fig, ax = plt.subplots(figsize=(10, 5))
fig.suptitle('Detection Horizon — JWST NIRSpec, 10 Transits', fontsize=13, fontweight='bold')
for name, snrs in horizon_snrs.items():
    ax.plot(distances_pc, snrs, '-o', color=COLORS[name], lw=2, ms=6, label=LABELS[name])
ax.axhline(5.0, color='black', lw=1.5, ls='--', label='5s detection threshold')
ax.axvline(12.43, color='purple', lw=1.0, ls='-.', alpha=0.7)
ax.text(12.8, max([s.max() for s in horizon_snrs.values()])*0.9, 'TRAPPIST-1\n12.4 pc', color='purple', fontsize=9)
ax.set_xlabel('Distance (pc)'); ax.set_ylabel('Broadband Detection SNR')
ax.legend(fontsize=9); ax.set_xlim(0, 52); ax.set_ylim(bottom=0)
plt.tight_layout()
plt.savefig('results/fig04_detection_horizon.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved results/fig04_detection_horizon.png')

## 6. Cloud Fraction Sensitivity

In [ ]:
cloud_fracs = [0.0, 0.2, 0.4, 0.6, 0.8, 0.95]
wl = default_wavelength_grid()
planet = PlanetSystem.trappist1e()
cmap = plt.cm.RdYlBu_r
snr_vs_cloud = []
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Cloud Fraction Sensitivity — Earth-like, TRAPPIST-1e, 10 transits', fontsize=13, fontweight='bold')
for cf in cloud_fracs:
    tmpl = build_earth_like_template(wl, cloud_fraction=cf, star_radius_rs=planet.star_radius_rs, planet_radius_re=planet.planet_radius_re)
    axes[0].plot(wl, tmpl.transit_depth_ppm, color=cmap(cf), lw=1.5, label=f'f_cloud={cf:.0%}')
    r = sim.simulate(planet, tmpl, n_transits=10)
    snr_vs_cloud.append(r.detection_snr)
axes[0].set_xlabel('Wavelength (um)'); axes[0].set_ylabel('Transit Depth (ppm)')
axes[0].set_title('Spectra vs. cloud fraction'); axes[0].legend(fontsize=8); axes[0].set_xlim(0.6, 5.3)
bars = axes[1].bar([f'{cf:.0%}' for cf in cloud_fracs], snr_vs_cloud, color=[cmap(cf) for cf in cloud_fracs], edgecolor='black', lw=0.8)
axes[1].axhline(5.0, color='black', lw=1.5, ls='--', label='5s threshold')
axes[1].set_xlabel('Cloud Fraction'); axes[1].set_ylabel('Broadband SNR')
axes[1].set_title('Detection SNR vs. cloud fraction'); axes[1].legend()
for bar, snr in zip(bars, snr_vs_cloud):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3, f'{snr:.1f}s', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig('results/fig05_cloud_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved results/fig05_cloud_sensitivity.png')

## Summary
Week 2 pipeline complete. Five figures generated in `results/`.

**Next steps (Week 3):** Retrieval framework — fit templates back to simulated data to recover atmospheric parameters.